In [1]:
import json
import os
import time
from google import genai
from google.genai import types
import datetime
import asyncio
import warnings

# Suppress the specific thought_signature warning from the SDK
warnings.filterwarnings("ignore", message=".*thought_signature.*")


with open('keys.json', 'r') as file:
    keys = json.load(file)

### Grounding the search for specific hardware EOS. 

In [4]:
with open('prompt.txt', 'r') as pmt_file:
    instruct = pmt_file.read()

In [5]:
def client_setup():
    client = genai.Client(api_key=keys['GEMINI_API_KEY'])

    # Set up the google search tool for the client.
    scraper_client = types.Tool(google_search=types.GoogleSearch())
    config = types.GenerateContentConfig(
        tools=[scraper_client],
        #thinking_config=types.ThinkingConfig(thinking_budget=-1), # swtich off if non-thinking model is used
        temperature = 0,
        top_p=1,
)
    return client, config

In [6]:
client, config = client_setup()

In [18]:
string = "Cisco IOS XE 16.12.08"

response = client.models.generate_content(
            model="gemini-2.5-flash",
            #contents=instruct_test + ' ' + string,
            contents=instruct + ' ' + string,
            #contents = string,
            config=config
        )

In [19]:
json_response = response.text
print(json_response)

```json
{
  "Name": "Cisco IOS XE 16.12.08",
  "Summary": "Cisco IOS XE 16.12.x reached its End-of-Sale on February 17, 2021, and its End-of-Support (Last Date of Support) is February 28, 2026.",
  "Hardware/Software": "Software",
  "Support Model": "Fixed",
  "EOS Date": "2026-02-28",
  "Support Tiers": [],
  "Source URLs": [
    "https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQFswcHf47uluwIfM3CEUl12M70cVpfRLtFxmTo4IXIYlnKiIIShY00v5AaiSnrjvoJqbxuqc8V4WHqUumQ9_cD_TAGqSFVIMCx7SJr9OOzQv4Ofj3DLOFLy90vqrUUqUUAU3qon17FJBhv4uUPr7t6FMCRVZ7qC5KuiGeJPKRdiq1Nu7tiCpXXf4zsNvmomu40vU1kWOPsHHLsJpt82JY9NH7KF_pT025o=",
    "https://vertexaisearch.cloud.google.com/grounding-api-redirect/AUZIYQHQSsJ4oOeYV56aUQFcVaQ90OsHy6o9RRtCSd3oon31__PIjF_1xIbqyZ3gjqOwCwirAKgTZfjjIY3mTNwXPHZ67sAvyDAA73Iqnq83DQ30xOwghDhWyRecSziOfAyo92Tehl2CqlFomKTWwVm-OKzRdNOvm62GnsjRk4yW2M5fYaCZlemUQ5rnkQ=="
  ],
  "Confidence": 1.00
}
```


In [5]:
from google.genai import types

# 1. Use the lowercase string 
thinking_setup = types.ThinkingConfig(
    thinking_level="low"  # Options: "minimal", "low", "medium", "high"
)

# 2. Add it to your main generation config
config = types.GenerateContentConfig(
    thinking_config=thinking_setup,
    temperature=0.0, 
    response_mime_type="application/json", 
)

string = "CISCO IOS XE 16.12.08"

# 3. Pass the config into your call just like you were doing
response = client.models.generate_content(
    model="gemini-3-flash-preview",
    contents=instruct + ' ' + string,
    config=config
)

# 4. Cleanly print the text, bypassing the thought_signature warning
json_response = ""
for part in response.candidates[0].content.parts:
    if part.text:
        json_response += part.text

print(json_response)

{
  "Name": "Cisco IOS XE 16.12.x (Amsterdam)",
  "Summary": "Cisco IOS XE 16.12.x reached its final milestone for software maintenance on July 31, 2022, with vulnerability support ending on July 31, 2024.",
  "Hardware/Software": "Software",
  "Support Model": "Version-Based",
  "EOS Date": "2024-07-31",
  "Support Tiers": [
    {
      "Tier": "End of SW Maintenance Releases",
      "EndDate": "2022-07-31"
    },
    {
      "Tier": "End of Vulnerability/Security Support",
      "EndDate": "2024-07-31"
    }
  ],
  "Source URLs": [
    "https://www.cisco.com/c/en/us/products/collateral/ios-nx-os-software/ios-xe-16/eos-eol-notice-c51-742899.html"
  ],
  "Confidence": 1.00
}


In [6]:
json_response = response.text
print(json_response)

{
  "Name": "Cisco IOS XE 16.12.x (Amsterdam)",
  "Summary": "Cisco IOS XE 16.12.x reached its final milestone for software maintenance on July 31, 2022, with vulnerability support ending on July 31, 2024.",
  "Hardware/Software": "Software",
  "Support Model": "Version-Based",
  "EOS Date": "2024-07-31",
  "Support Tiers": [
    {
      "Tier": "End of SW Maintenance Releases",
      "EndDate": "2022-07-31"
    },
    {
      "Tier": "End of Vulnerability/Security Support",
      "EndDate": "2024-07-31"
    }
  ],
  "Source URLs": [
    "https://www.cisco.com/c/en/us/products/collateral/ios-nx-os-software/ios-xe-16/eos-eol-notice-c51-742899.html"
  ],
  "Confidence": 1.00
}


### Processing the HW File

In [4]:
import pandas as pd
import time
from classes import Helper, Cleaner, Processing

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:


df = Helper.asset_list_cleaning('SWandHW.xlsx')

Number of hardware items: 1
Number of software items: 0


In [6]:
df

(['iPhone 17 Pro Max'], [])

### Gemini API and JSON parsing

In [9]:
import numpy as np
winnie_sample = np.array(['Nutanix AOS 6.5.4 LTS', 'Ubuntu 24.04.1 LTS', 'Window 11 Pro 24H2', 'Windows Server 2022 DataCenter'])


In [ ]:
import asyncio
from classes import Helper, Cleaner, Processing

def error_cache(results, eos_list):
    """
    Caches the results and eos_list for items that failed to process.
    """
    success, unsuccess = [], []
    for i, result in enumerate(results):
        if result is not None:
            success.append(result)
        else:
            unsuccess.append(eos_list[i])  # This mapping is always correct
    return success, unsuccess

async def process_line(string, client, config):
    print(f"Processing item: {string}")
    try:
        await asyncio.sleep(5)  # Sleep to avoid hitting rate limits
        response = client.models.generate_content(
        model="gemini-3-flash-preview",
        contents=instruct + ' ' + string,
        config=config
    )
        json_response = "" #
        for part in response.candidates[0].content.parts:
            if part.text:
                json_response += part.text
        return Helper.parse_llm_json(json_response) # includes throwing none in here as well
    except Exception as e:
        print(f"Error processing {string}: {e}")
        return None # need to add some handling here to log if needed. 

async def main(eos_list):
    client, config = client_setup()
    tasks = [process_line(item, client, config) for item in eos_list]
    results = await asyncio.gather(*tasks)

    # error caching for failed API calls or bad responses
    success, unsuccess = error_cache(results, eos_list)

    print(f"Successfully processed {len(success)} items.")
    
    # Add a retry limit if needed
    retry_limit = 3
    retry_count = 0

    while unsuccess and retry_count < retry_limit:
        print(f"Retrying {len(unsuccess)} items...")
        retry_tasks = [process_line(item, client, config) for item in unsuccess]
        retry_results = await asyncio.gather(*retry_tasks)
        retry_success, unsuccess = error_cache(retry_results, unsuccess)
        
        # add to main success list
        success.extend(retry_success)

        retry_count += 1
        if unsuccess:
            print(f"Retry {retry_count} failed for {len(unsuccess)} items.")
    
    return success, unsuccess

async def run_async(lst):
    print("Starting async processing...")
    # add time start
    start_time = time.time()
    results, failed_items = await main(lst)
    # add time end
    elapsed = time.time() - start_time  # Calculate elapsed time
    print(f"Time taken: {elapsed:.2f} seconds")  # Print elapsed time
    print("Async processing completed.")
    return results, failed_items 

# Run the async function, change this for the script. 
results, failed_items = await run_async(winnie_sample)
results


In [12]:
import pandas as pd
def processing_tiers(results):
    df = pd.DataFrame(results)

    # 2. Explode the 'Support Tiers' column to create separate rows
    #    Products with no tiers (like Chrome) will result in a row with NaN
    df_exploded = df.explode('Support Tiers').reset_index(drop=True)

    # 3. Normalize the 'Support Tiers' column (which now contains dictionaries)
    #    and join it back to the main data
    tiers_df = pd.json_normalize(df_exploded['Support Tiers'])
    final_df = df_exploded.drop(columns=['Support Tiers']).join(tiers_df)

    # Optional: Convert date strings to datetime objects for calculations
    final_df['EOS Date'] = pd.to_datetime(final_df['EOS Date'])
    final_df['EndDate'] = pd.to_datetime(final_df['EndDate'])
    return final_df
final_df = processing_tiers(results)
final_df.to_excel('winnie_sample.xlsx', index=False)

In [ ]:
from classes import Helper
helper = Helper()
helper.preprocess('SWandHW.xlsx', sheet='Sheet1')

# test cases



Starting Preprocessing for: SWandHW.xlsx...
Number of hardware items: 1
Number of software items: 0
Processing completed in 19.67s
-----------------------------------


(['iPhone 17 Pro Max'], [])